In [0]:
SELECT *
FROM workspace.default.customers ;

SELECT *
FROM workspace.default.orders ;

SELECT *
FROM workspace.default.products ;

SELECT *
FROM workspace.default.sales_reps ;

SELECT *
FROM workspace.default.suppliers ;

***QUESTION_ONE***
---**The Head of Sales has requested a revenue performance report segmented by customer tier. She needs
to understand which customer segments are driving the most revenue from completed orders only.**---

---**Business Scenario:
The Head of Sales has requested a revenue performance report segmented by customer tier. She needs
to understand which customer segments are driving the most revenue from completed orders only.**---

SELECT 
       a.segment,
       b.status,
       COUNT(b.order_id) AS total_orders,
       SUM(b.unit_price * b.quantity) AS total_revenue
FROM workspace.default.customers AS a
INNER JOIN workspace.default.orders AS b
ON a.customer_id = b.customer_id
GROUP BY a.segment, b.status
HAVING b.status = 'Completed'
ORDER BY total_revenue DESC ;

---***Question_Two***---
--**The Customer Success team is running a re-engagement campaign. They need a list of all registered
customers who have never placed an order so they can reach out proactively.**--

SELECT a.customer_id,
       a.customer_name,
       a.region,
       a.segment,
       b.order_id
FROM workspace.default.customers AS a
LEFT JOIN workspace.default.orders AS b
ON a.customer_id = b.customer_id
WHERE b.order_id IS NULL 
ORDER BY customer_id;

---***QUESTION_THREE***---
---**An audit has revealed that some Enterprise-tier customers may not have an assigned sales
representative, which is a breach of the company's key account policy. The VP of Sales needs an
immediate list.**---

SELECT a.customer_id,
       a.customer_name,
       a.region,
       a.segment,
       b.rep_id,
       COALESCE (b.rep_name,'UNASSIGNED') AS assigned_rep
FROM workspace.default.customers AS a
LEFT JOIN workspace.default.sales_reps AS b
ON a.customer_id = b.customer_id
WHERE a.segment = 'Enterprise'
ORDER BY assigned_rep, a.customer_id;

---**QUESTION_FOUR**--
---**The Finance team is building a product P&L dashboard. They need to see, at the order line level, the
revenue, cost, and gross profit for every completed order, along with the product name and category.**--

SELECT b.order_id,
       b.order_date,
       b.product_id,
       c.product_name,
       c.category,
       SUM(b.unit_price * b.quantity) AS total_revenue,
       SUM(c.cost_price * b.quantity) AS total_cost,
       SUM(b.unit_price * b.quantity) - SUM(c.cost_price * b.quantity) AS gross_profit
FROM workspace.default.orders AS b
INNER JOIN workspace.default.products AS c
ON b.product_id = c.product_id
WHERE b.status = 'Completed'
GROUP BY b.order_id,
         b.order_date,
         b.product_id,
         c.product_name,
         c.category
ORDER BY gross_profit DESC ;

---***QUESTION_FIVE***---
--**Business Scenario:
The Procurement team suspects there are products linked to inactive or missing suppliers, AND suppliers
on record who have no products assigned to them. A full reconciliation report is required before the annual
vendor review.**--

SELECT c.product_name,
       c.supplier_id,
       e.supplier_name,
       e.contract_status,
    CASE 
    WHEN c.supplier_id IS NULL THEN 'NO SUPPLIER'
    WHEN c.supplier_id IS NOT NULL AND c.product_name IS NULL THEN 'NO PRODUCTS'
    ELSE 'OK'
    END AS reconciliation_flag
FROM workspace.default.products AS c
FULL OUTER JOIN workspace.default.suppliers AS e
ON c.supplier_id = e.supplier_id
ORDER BY reconciliation_flag, c.product_name  ;

---**QUESTION_SIX**--
---**Business Scenario:
The Data Science team is building a customer scoring model and needs a base table showing every
customer alongside their total lifetime revenue and number of completed orders. Customers with zero
completed orders must still appear with zeroes — not be excluded.**---

SELECT a.customer_id,
       a.customer_name,
       a.region,
       a.segment,
       b.status,
       COALESCE(COUNT(CASE WHEN b.status = 'Completed' THEN b.order_id END), 0) AS total_orders,
       COALESCE(SUM(CASE WHEN b.status = 'Completed' THEN b.quantity * b.unit_price END), 0) AS lifetime_value
FROM workspace.default.customers AS a
LEFT JOIN workspace.default.orders AS b
ON a.customer_id = b.customer_id
GROUP BY a.customer_id,
         a.customer_name,
         a.region,
         a.segment,
         b.status
ORDER BY lifetime_value DESC ;

---**QUESTION_SEVEN**--
---**Business Scenario:
The CFO has flagged anomalies in the orders table: some orders reference customer IDs that do not exist
in the customers table (orphaned orders), and some customers have never ordered. Additionally, the CFO
wants to see the assigned sales rep for each customer where one exists. This query will feed into the
month-end audit report.**---

SELECT a.customer_id,
       a.customer_name,
       b.order_id,
       b.order_date,
       b.status,
COALESCE(d.rep_name, 'NO REP') AS rep_name,
CASE WHEN a.customer_id IS NULL THEN 'ORPHANED ORDER'
    WHEN b.order_id IS NULL THEN 'NO ORDERS'
    ELSE 'OK'
    END AS audit_flag
FROM workspace.default.customers AS a
FULL OUTER JOIN orders AS b
ON a.customer_id = b.customer_id
LEFT JOIN sales_reps AS d
ON a.customer_id = d.customer_id
ORDER BY audit_flag ; 